# APScheduler

APScheduler 是应用级定时任务工具，适合后台同步、报表生成、定时清理和轻量任务编排。它支持 interval、date、cron 等触发器，以及线程池/进程池执行器。


## 基本配置


In [ ]:
from apscheduler.executors.pool import ProcessPoolExecutor, ThreadPoolExecutor
from apscheduler.jobstores.memory import MemoryJobStore
from apscheduler.schedulers.background import BackgroundScheduler

scheduler = BackgroundScheduler(
    timezone='Asia/Shanghai',
    jobstores={'default': MemoryJobStore()},
    executors={
        'default': ThreadPoolExecutor(max_workers=10),
        'processpool': ProcessPoolExecutor(max_workers=2),
    },
    job_defaults={
        'coalesce': True,
        'max_instances': 1,
        'misfire_grace_time': 30,
    },
)


## Trigger 示例


In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from apscheduler.triggers.cron import CronTrigger
from apscheduler.triggers.date import DateTrigger
from apscheduler.triggers.interval import IntervalTrigger

SHANGHAI = ZoneInfo('Asia/Shanghai')


def send_report(report_name: str) -> str:
    return f'{report_name} sent at {datetime.now(SHANGHAI):%Y-%m-%d %H:%M:%S}'


scheduler.add_job(
    send_report,
    trigger=IntervalTrigger(seconds=10),
    args=['interval-report'],
    id='interval_report',
    replace_existing=True,
)

scheduler.add_job(
    send_report,
    trigger=DateTrigger(run_date=datetime.now(SHANGHAI) + timedelta(minutes=5)),
    args=['once-report'],
    id='once_report',
    replace_existing=True,
)

scheduler.add_job(
    send_report,
    trigger=CronTrigger(hour=9, minute=30, timezone=SHANGHAI),
    args=['daily-report'],
    id='daily_report',
    replace_existing=True,
)


## 管理任务


In [ ]:
def refresh_cache(cache_name: str) -> str:
    return f'{cache_name} refreshed'


scheduler.add_job(refresh_cache, 'interval', seconds=30, args=['user_profile'], id='refresh_cache')
job_ref = scheduler.get_job('refresh_cache')
print('Job:', job_ref)

scheduler.modify_job('refresh_cache', seconds=60)
scheduler.pause_job('refresh_cache')
scheduler.resume_job('refresh_cache')
scheduler.remove_job('refresh_cache')


## 监听器与生命周期


In [ ]:
from apscheduler.events import EVENT_JOB_ERROR, EVENT_JOB_EXECUTED, JobExecutionEvent


def log_job_event(event: JobExecutionEvent) -> None:
    if event.exception:
        print(f'Job failed: {event.job_id}')
    else:
        print(f'Job succeeded: {event.job_id}')


scheduler.add_listener(log_job_event, EVENT_JOB_EXECUTED | EVENT_JOB_ERROR)
scheduler.print_jobs()

if not scheduler.running:
    scheduler.start(paused=True)

scheduler.resume()
scheduler.pause()
scheduler.shutdown(wait=False)


## 应用级封装


In [ ]:
from collections.abc import Callable

from apscheduler.schedulers.background import BackgroundScheduler


class SchedulerService:
    def __init__(self, scheduler: BackgroundScheduler) -> None:
        self._scheduler = scheduler

    def add_interval_job(
        self,
        job_id: str,
        func: Callable[..., object],
        seconds: int,
        *args: object,
    ) -> None:
        self._scheduler.add_job(
            func,
            trigger='interval',
            seconds=seconds,
            args=args,
            id=job_id,
            replace_existing=True,
        )

    def start(self) -> None:
        if not self._scheduler.running:
            self._scheduler.start()

    def stop(self) -> None:
        if self._scheduler.running:
            self._scheduler.shutdown(wait=False)
